In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely.geometry as sg
import folium
import base64
import requests
import json 
import time

In [2]:
from xyzservices import TileProvider
from zwerfafval_detectie.utils_eval import read_annotations_folder


RD_CRS = "EPSG:28992"  # CRS code for the Dutch Rijksdriehoek coordinate system
LAT_LON_CRS = "EPSG:4326"  # CRS code for WGS84 latitude/longitude coordinate system

ams_tile_provider = TileProvider(
    name="Topografie, standaard visualisatie (WM)",
    url="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attribution="data.amsterdam.nl",
)

In [3]:
model = "yolo26m_1920_v1-2_extra_250-2"
split = "train"

predictions_folder = f"../datasets/experiments/zwerfafval/predict/{model}/{split}"

categories = {
    0: "Zwerfafval (grof)",
    1: "Zwerfafval (fijn)"
}

confidence = 0.2 #confidence threshold for yolo26

In [4]:
predictions_gdf = read_annotations_folder(folder_path=predictions_folder, categories=categories)
predictions_gdf["file_name"] = predictions_gdf["file_name"].str.replace(".txt", ".jpg")

_predictions_sorted = (
    predictions_gdf[predictions_gdf["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)

counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)

In [5]:
metadata_files = [
    "../datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300.gpkg",
    "../datasets/experiments/zwerfafval/annotatieproject/inwinning_260421_selectie_1000.gpkg"
]

metadata_gdf = pd.concat([
    gpd.read_file(metadata_file, layer=0)
    for metadata_file in metadata_files
]).set_index("file_name")

In [6]:
counts_merged = gpd.GeoDataFrame(counts_df.join(metadata_gdf, how="left"))
counts_merged = counts_merged[["Zwerfafval (fijn)", "Zwerfafval (grof)", "geometry"]].to_crs(RD_CRS).dropna()

In [7]:
# Create map with different radii
map = (
    counts_merged
    .explore(
        column="Zwerfafval (grof)",
        cmap="YlOrRd",
        style_kwds={
            "style_function": lambda x: {"radius": 2*x["properties"]["Zwerfafval (grof)"]},
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

map.save(os.path.join("../datasets/experiments/zwerfafval", f"heatmap_v1_{model}_{split}.html"))

In [13]:
counts_merged

,Zwerfafval (fijn),Zwerfafval (grof),geometry
file_name,,,
20250514_194817_711097_000511.jpg,3,2,POINT (121449.197 485076.852)
20250514_194824_789533_000623.jpg,17,5,POINT (121451.399 485071.953)
20250514_194826_554034_000651.jpg,11,1,POINT (121452.572 485066.148)
20250514_194827_871855_000672.jpg,6,1,POINT (121453.264 485060.735)
20250514_194831_874718_000735.jpg,4,2,POINT (121454.223 485043.46)
...,...,...,...
20250514_200626_221499_017745.jpg,4,1,POINT (120926.994 485363.026)
20250514_200627_512295_017766.jpg,3,1,POINT (120927.609 485357.303)
20250514_200628_869633_017787.jpg,1,2,POINT (120928.096 485350.823)


In [9]:
# total area for the grid
xmin, ymin, xmax, ymax = counts_merged.total_bounds

xmin = int(xmin / 100) * 100
ymin = int(ymin / 100) * 100
xmax = int((xmax + 100) / 100) * 100
ymax = int((ymax + 100) / 100) * 100

# how many cells across and down
cell_size = 100

# create the cells in a loop
grid_cells = []
for x0 in np.arange(xmin, xmax+cell_size, cell_size ):
    for y0 in np.arange(ymin, ymax+cell_size, cell_size):
        # bounds
        x1 = x0-cell_size
        y1 = y0+cell_size
        grid_cells.append( sg.box(x0, y0, x1, y1)  )
grid = gpd.GeoDataFrame(grid_cells, columns=['geometry'], crs=RD_CRS)

merged = gpd.sjoin(counts_merged, grid, how='left', predicate='within')

dissolve = merged.dissolve(by="index_right", aggfunc="sum")

grid.loc[dissolve.index, "Zwerfafval (grof)"] = dissolve["Zwerfafval (grof)"].values
grid = grid.dropna()

In [10]:
# Create map with grid
map = (
    grid
    .explore(
        column="Zwerfafval (grof)",
        cmap="YlOrRd",
        style_kwds={
            "fillOpacity": 0.75,
            "weight": 2
        },
        legend=True,
        tiles=ams_tile_provider
    )
)

map.save(os.path.join("../datasets/experiments/zwerfafval", f"heatmap_v1_{model}_{split}_grid.html"))

In [11]:
# Option 1: Heatmap (dynamic kernel density estimation (KDE)/space-time cube heatmap)
#------------------------------------------------

from folium.plugins import MarkerCluster, HeatMap

# Start from grid explore map
map_cluster = grid.explore(
    column="Zwerfafval (grof)",
    cmap="YlOrRd",
    style_kwds={"fillOpacity": 0.55, "weight": 0.5},
    legend=True,
    tiles=ams_tile_provider
)

# Convert points to WGS84 for Folium
points_wgs = counts_merged.to_crs("EPSG:4326")

# Marker cluster layer
cluster = MarkerCluster(name="Detecties").add_to(map_cluster)
for _, row in points_wgs.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="#E24B4A",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Grof: {row['Zwerfafval (grof)']} | Fijn: {row['Zwerfafval (fijn)']}"
    ).add_to(cluster)

# Folium layer
heat_data = [
    [row.geometry.y, row.geometry.x, row["Zwerfafval (grof)"]]
    for _, row in points_wgs.iterrows()
    if row["Zwerfafval (grof)"] > 0
]
HeatMap(heat_data, name="Heatmap", radius=18, blur=12, max_zoom=15).add_to(map_cluster)

folium.LayerControl().add_to(map_cluster)
map_cluster.save(f"../datasets/experiments/zwerfafval/heatmap_cluster_{model}_{split}.html")

In [12]:
# Option 2: Fixed-radius dots, color = density
#------------------------------------------------

from folium import CircleMarker
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

#data
points = counts_merged.copy()
points["total"] = points["Zwerfafval (fijn)"] + points["Zwerfafval (grof)"]
points = points[points["total"] > 0].copy()
points_wgs = points.to_crs("EPSG:4326")

#Color scale
cmap     = plt.get_cmap("YlOrRd")
log_vals = np.log1p(points_wgs["total"].values.astype(float))
norm     = mcolors.Normalize(vmin=log_vals.min(), vmax=log_vals.max())

def density_to_hex(val):
    return mcolors.to_hex(cmap(norm(np.log1p(val))))


#Create map
center = [points_wgs.geometry.y.mean(), points_wgs.geometry.x.mean()]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
    name="Amsterdam Topo",
).add_to(m)

for _, row in points_wgs.iterrows():
    total = int(row["total"])
    grof  = int(row["Zwerfafval (grof)"])
    fijn  = int(row["Zwerfafval (fijn)"])
    color = density_to_hex(total)

    CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=0.8,
        tooltip=folium.Tooltip(
            f"""<div style="font-family:system-ui;font-size:12px">
              <b>Totaal:</b> {total} detecties<br>
              <b>Grof:</b> {grof} &nbsp;|&nbsp; <b>Fijn:</b> {fijn}
            </div>""",
            sticky=True,
        ),
    ).add_to(m)


legend_html = """
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px">
  <div style="font-weight:600;margin-bottom:8px;color:#111">Detectiedichtheid</div>
  <div style="width:120px;height:14px;border-radius:3px;
    background:linear-gradient(to right,#ffffb2,#fecc5c,#fd8d3c,#f03b20,#bd0026)"></div>
  <div style="display:flex;justify-content:space-between;width:120px;margin-top:3px;color:#666">
    <span>Laag</span><span>Hoog</span>
  </div>
  <div style="margin-top:8px;color:#888;font-size:10px">Radius = vast (6px) | Kleur = log(fijn + grof)</div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))


out_path = os.path.join(
    "../datasets/experiments/zwerfafval",
    f"heatmap_dots_{model}_{split}.html"
)
m.save(out_path)
print(f"Saved → {out_path}  ({len(points_wgs)} dots)")

Saved → ../datasets/experiments/zwerfafval/heatmap_dots_yolo26m_1920_v1-2_extra_250-2_train.html  (160 dots)


In [ ]:
# Option 3: Fixed-radius dots, two category layers, hover over point: show image, street name, & datetime
from folium import CircleMarker
from folium.plugins import GroupedLayerControl
from datetime import datetime

# Data points and convert to WGS84 for mapping coordinates
points = counts_merged.copy()
points["total"] = points["Zwerfafval (fijn)"] + points["Zwerfafval (grof)"]
points = points[points["total"] > 0].copy()
points_wgs = points.to_crs("EPSG:4326")

# 2. Download official Amsterdam Streetnames LineStrings GeoJSON
print("Downloading Amsterdam street names GeoJSON...")
street_url = "https://maps.amsterdam.nl/open_geodata/geojson_lnglat.php?KAARTLAAG=STRAATNAMEN&THEMA=straatnamen"
response = requests.get(street_url)
streets_gdf = gpd.GeoDataFrame.from_features(response.json()["features"], crs="EPSG:4326")

# Use column 'STT_NAAM' which is used in Amsterdam's open data map layer
name_col = next((col for col in ['STT_NAAM', 'Naam', 'straatnaam', 'name', 'STRAATNAAM'] if col in streets_gdf.columns), streets_gdf.columns[0])
print(f"Using street name column: {name_col}")

# Couple each point to the closest street LineString using metric projection (EPSG:28992)
print("Coupling each point to the closest street line...")
points_m = points_wgs.to_crs("EPSG:28992")
streets_m = streets_gdf.to_crs("EPSG:28992")

joined = gpd.sjoin_nearest(points_m, streets_m, how="left", distance_col="dist_to_street")

# Assign the nearest found street name back to points_wgs
points_wgs["street"] = joined[name_col].values

# Image lookup indexing
image_folders = [
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300",
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_1000",
]

def build_image_index(folders):
    index = {}
    for folder in folders:
        if not os.path.isdir(folder):
            print(f"  ⚠ Folder not found: {folder}")
            continue
        for fname in os.listdir(folder):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                key = os.path.splitext(fname)[0]
                index[key] = os.path.join(folder, fname)
    return index

print("Indexing images...")
image_index = build_image_index(image_folders)
print(f"  Found {len(image_index)} images")

def get_image_b64(file_name):
    key = os.path.splitext(os.path.basename(file_name))[0]
    path = image_index.get(key)
    if path and os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        ext = os.path.splitext(path)[1].lower().replace(".jpg", ".jpeg")
        return f"data:image/{ext.strip('.')};base64,{data}"
    return None

# 5. Linear Color Scales per category
def make_norm_hex(series, cmap_name):
    cmap  = plt.get_cmap(cmap_name)
    min_v = float(series.values.min())
    max_v = float(series.values.max())
    if max_v == min_v:
        max_v = min_v + 1.0
    norm  = mcolors.Normalize(vmin=min_v, vmax=max_v)
    def to_hex(val):
        return mcolors.to_hex(cmap(norm(float(val))))
    return to_hex, int(min_v), int(max_v)

to_hex_grof, min_grof, max_grof = make_norm_hex(points_wgs["Zwerfafval (grof)"], "YlOrRd")
to_hex_fijn, min_fijn, max_fijn = make_norm_hex(points_wgs["Zwerfafval (fijn)"], "YlGnBu")

# 6. Create Map
center = [points_wgs.geometry.y.mean(), points_wgs.geometry.x.mean()]
m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
    name="Amsterdam Topo",
).add_to(m)

layer_grof = folium.FeatureGroup(name="Zwerfafval (grof)", show=True)
layer_fijn = folium.FeatureGroup(name="Zwerfafval (fijn)", show=True)

missing_images = 0

for idx, (file_name, row) in enumerate(points_wgs.iterrows()):
    lat   = row.geometry.y
    lon   = row.geometry.x
    grof  = int(row["Zwerfafval (grof)"])
    fijn  = int(row["Zwerfafval (fijn)"])
    total = int(row["total"])
    
    # Retrieve the dynamically mapped street name
    street = row.get("street", "Onbekende straat")
    if not isinstance(street, str) or not street.strip() or street == "nan":
        street = "Onbekende straat"
        
    # Check possible timestamp column names
    datetime_str = row.get("datetime", row.get("tijd", row.get("date", row.get("timestamp", "Onbekende tijd"))))

    # Embed bigger image (1080x720px)
    img_b64 = get_image_b64(file_name)
    if img_b64:
        img_html = f'<img src="{img_b64}" style="width:1080px;height:720px;object-fit:cover;border-radius:4px;display:block;margin-bottom:6px">'
    else:
        missing_images += 1
        img_html = '<div style="width:1080px;height:720px;background:#f0f0ee;border-radius:4px;display:flex;align-items:center;justify-content:center;color:#aaa;font-size:12px;margin-bottom:6px">Geen afbeelding</div>'

    fname_label = os.path.basename(file_name)

    def make_tooltip(count, color, category):
        return folium.Tooltip(
            f"""<div style="font-family:system-ui;font-size:12px;max-width:380px">
              {img_html}
              <div style="font-size:10px;color:#888;margin-bottom:4px;word-break:break-all"><b>Bestand:</b> {fname_label}</div>
              <div style="font-size:11px;color:#333;margin-bottom:2px">📍 <b>Straat:</b> {street}</div>
              <div style="font-size:11px;color:#333;margin-bottom:4px">🕒 <b>Tijdstip:</b> {datetime_str}</div>
              <div style="display:flex;align-items:center;gap:6px;margin-bottom:2px">
                <span style="display:inline-block;width:10px;height:10px;border-radius:2px;background:{color}"></span>
                <b>{category}:</b> {count}
              </div>
              <div style="color:#555"><b>Totaal:</b> {total} &nbsp;|&nbsp;
                <b>Grof:</b> {grof} &nbsp;|&nbsp; <b>Fijn:</b> {fijn}
              </div>
            </div>""",
            sticky=True,
        )

    if grof > 0:
        color_grof = to_hex_grof(grof)
        CircleMarker(
            location=[lat, lon],
            radius=5,
            color=color_grof,
            fill=True,
            fill_color=color_grof,
            fill_opacity=0.7,
            weight=0.8,
            tooltip=make_tooltip(grof, color_grof, "Grof"),
        ).add_to(layer_grof)

    if fijn > 0:
        color_fijn = to_hex_fijn(fijn)
        CircleMarker(
            location=[lat + 0.00003, lon + 0.00003],
            radius=5,
            color=color_fijn,
            fill=True,
            fill_color=color_fijn,
            fill_opacity=0.7,
            weight=0.8,
            tooltip=make_tooltip(fijn, color_fijn, "Fijn"),
        ).add_to(layer_fijn)

layer_grof.add_to(m)
layer_fijn.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

print(f"  {missing_images} points had no matching image")

# 7. Linear Scale Legend
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px;min-width:180px">
  <div style="font-weight:600;margin-bottom:10px;color:#111">Detectiedichtheid (Lineair)</div>

  <div style="margin-bottom:6px;font-size:11px;color:#444;font-weight:600">Grof</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#ffffb2,#fecc5c,#fd8d3c,#f03b20,#bd0026)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:2px;color:#888;font-size:10px">
    <span>{min_grof}</span><span>{max_grof}</span>
  </div>

  <div style="margin:10px 0 6px;font-size:11px;color:#444;font-weight:600">Fijn</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#ffffd9,#c7e9b4,#41b6c4,#2c7fb8,#253494)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:2px;color:#888;font-size:10px">
    <span>{min_fijn}</span><span>{max_fijn}</span>
  </div>

  <div style="margin-top:10px;color:#aaa;font-size:10px">Radius = vast (6px)<br>Kleur = lineaire telling</div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

# 8. Save output
out_path = os.path.join(
    "../datasets/experiments/zwerfafval",
    f"heatmap_dots_{model}_{split}.html"
)
m.save(out_path)
print(f"Saved → {out_path}  ({len(points_wgs)} points)")

Using street name column: STT_NAAM
Coupling each point to the closest street line...
Indexing images...
  ⚠ Folder not found: /home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_1000
  Found 300 images
  0 points had no matching image
Saved → ../datasets/experiments/zwerfafval/heatmap_dots_yolo26m_1920_v1-2_extra_250-2_train.html  (160 points)


In [ ]:
# ── 1. Load GPS JSON ──────────────────────────────────────────────────────────

GPS_JSON_PATH = "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/gps_info_20250514_200648.json"

with open(GPS_JSON_PATH, "r") as f:
    gps_raw = json.load(f)

# Build lookup: image_file_name (no ext) → {datetime, lat, lon}
gps_lookup = {}
for frame in gps_raw["frames"]:
    key = os.path.splitext(frame["image_file_name"])[0]
    ts  = frame.get("image_file_timestamp") or frame.get("record_timestamp")
    gps_lookup[key] = {
        "datetime": ts,
        "lat": frame["gps_data"]["latitude"],
        "lon": frame["gps_data"]["longitude"],
    }

print(f"GPS lookup: {len(gps_lookup)} entries")

# ── 2. Prepare detection data ─────────────────────────────────────────────────

points = counts_merged.copy()
points["total"] = points["Zwerfafval (fijn)"] + points["Zwerfafval (grof)"]
points = points[points["total"] > 0].copy()
points_wgs = points.to_crs("EPSG:4326")

# ── 3. Image folders ──────────────────────────────────────────────────────────

IMAGE_FOLDERS = [
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_300",
    "/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_1000",
]

def build_image_index(folders):
    index = {}
    for folder in folders:
        if not os.path.isdir(folder):
            print(f"  ⚠ Folder not found: {folder}")
            continue
        for fname in os.listdir(folder):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                key = os.path.splitext(fname)[0]
                index[key] = os.path.join(folder, fname)
    return index

print("Indexing images...")
image_index = build_image_index(IMAGE_FOLDERS)
print(f"  Found {len(image_index)} images")

def get_image_b64(file_name):
    key  = os.path.splitext(os.path.basename(file_name))[0]
    path = image_index.get(key)
    if path and os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        ext = os.path.splitext(path)[1].lower()
        mime = "jpeg" if ext in (".jpg", ".jpeg") else ext.strip(".")
        return f"data:image/{mime};base64,{data}"
    return None

# ── 4. Pre-fetch street names via Nominatim ───────────────────────────────────

CACHE_PATH = os.path.join(
    "../datasets/experiments/zwerfafval", "street_cache.json"
)
street_cache = json.loads(open(CACHE_PATH).read()) if os.path.isfile(CACHE_PATH) else {}

def get_street_name(lat, lon):
    cache_key = f"{lat:.5f},{lon:.5f}"
    if cache_key in street_cache:
        return street_cache[cache_key]
    try:
        r = requests.get(
            "https://nominatim.openstreetmap.org/reverse",
            params={"lat": lat, "lon": lon, "format": "json", "zoom": 17, "addressdetails": 1},
            headers={"User-Agent": "zwerfafval-heatmap/1.0"},
            timeout=5,
        )
        if r.status_code == 200:
            a = r.json().get("address", {})
            name = (a.get("road") or a.get("pedestrian")
                    or a.get("footway") or a.get("cycleway") or "Onbekend")
        else:
            name = "Onbekend"
    except Exception:
        name = "Onbekend"
    street_cache[cache_key] = name
    with open(CACHE_PATH, "w") as f:
        json.dump(street_cache, f)
    time.sleep(1.1)   # Nominatim rate limit
    return name

print(f"Fetching street names ({len(points_wgs)} points, cached={len(street_cache)})...")
street_names = {}
for i, (file_name, row) in enumerate(points_wgs.iterrows()):
    street_names[file_name] = get_street_name(row.geometry.y, row.geometry.x)
    if i % 10 == 0:
        print(f"  {i}/{len(points_wgs)}")
print("Done.")


GPS lookup: 2584 entries
Indexing images...
  ⚠ Folder not found: /home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_250514_selectie_1000
  Found 300 images
Fetching street names (160 points, cached=160)...
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
Done.


In [30]:
# ── 5. Format datetime helper ─────────────────────────────────────────────────

from datetime import datetime

def fmt_dt(ts_str):
    if not ts_str:
        return None, None

    try:
        # Handle trailing Z if present
        ts_str = ts_str.replace("Z", "+00:00")

        dt = datetime.fromisoformat(ts_str)

        return (
            dt.strftime("%d-%m-%Y"),
            dt.strftime("%H:%M:%S")
        )

    except Exception as e:
        print("Datetime parse error:", ts_str, e)
        return None, None

# ── 6. Color scales ───────────────────────────────────────────────────────────

def make_to_hex(series, cmap_name):
    cmap  = plt.get_cmap(cmap_name)
    log_v = np.log1p(series.values.astype(float))
    norm  = mcolors.Normalize(vmin=log_v.min(), vmax=max(log_v.max(), 1))
    def to_hex(val):
        return mcolors.to_hex(cmap(norm(np.log1p(val))))
    return to_hex

to_hex_grof = make_to_hex(points_wgs["Zwerfafval (grof)"], "YlOrRd")
to_hex_fijn = make_to_hex(points_wgs["Zwerfafval (fijn)"], "YlGnBu")

In [31]:
# ── 7. Build map ──────────────────────────────────────────────────────────────

center = [points_wgs.geometry.y.mean(), points_wgs.geometry.x.mean()]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
).add_to(m)

layer_grof = folium.FeatureGroup(name="Zwerfafval (grof)", show=True)
layer_fijn = folium.FeatureGroup(name="Zwerfafval (fijn)", show=True)

missing_img = 0

for file_name, row in points_wgs.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    grof = int(row["Zwerfafval (grof)"])
    fijn = int(row["Zwerfafval (fijn)"])
    total = int(row["total"])

    # GPS metadata
    key = os.path.splitext(os.path.basename(file_name))[0]
    gps_meta = gps_lookup.get(key, {})
    dt_date, dt_time = fmt_dt(gps_meta.get("datetime"))
    street = street_names.get(file_name, "Onbekend")

    # Image
    img_b64 = get_image_b64(file_name)
    if img_b64:
        img_html = (
            f'<img src="{img_b64}" '
            f'style="width:260px;height:170px;object-fit:cover;'
            f'border-radius:5px;display:block;margin-bottom:8px">'
        )
    else:
        missing_img += 1
        img_html = (
            '<div style="width:260px;height:60px;background:#f0f0ee;'
            'border-radius:5px;display:flex;align-items:center;'
            'justify-content:center;color:#bbb;font-size:11px;'
            'margin-bottom:8px">'
            'Geen afbeelding'
            '</div>'
        )

    def make_tooltip(count, color, category):
        badge = (
            f'<span style="display:inline-block;'
            f'background:{color};color:white;'
            f'border-radius:4px;padding:1px 7px;'
            f'font-size:10px;font-weight:600;'
            f'margin-bottom:6px">{category}</span>'
        )

        return folium.Tooltip(
            f"""
            <div style="font-family:system-ui;font-size:12px;max-width:270px">
              {img_html}
              {badge}
              <table style="width:100%;border-collapse:collapse;line-height:1.6">
                <tr>
                  <td style="color:#888;padding-right:8px;white-space:nowrap">📍 Straat</td>
                  <td style="font-weight:600;color:#111">{street}</td>
                </tr>
                <tr>
                  <td style="color:#888;white-space:nowrap">📅 Datum</td>
                  <td style="color:#444">{dt_date or "Onbekend"}</td>
                </tr>
                <tr>
                  <td style="color:#888;white-space:nowrap">🕐 Tijd</td>
                  <td style="color:#444">{dt_time or "Onbekend"}</td>
                </tr>
                <tr>
                  <td style="color:#888;white-space:nowrap">📦 {category}</td>
                  <td style="color:#444">{count} detecties</td>
                </tr>
                <tr>
                  <td style="color:#888;white-space:nowrap">Totaal</td>
                  <td style="color:#444">
                    Grof {grof} &nbsp;|&nbsp; Fijn {fijn}
                  </td>
                </tr>
              </table>
            </div>
            """,
            sticky=True,
        )

    if grof > 0:
        color_grof = to_hex_grof(grof)
        CircleMarker(
            location=[lat, lon],
            radius=6,
            color=color_grof,
            fill=True,
            fill_color=color_grof,
            fill_opacity=0.85,
            weight=0.8,
            tooltip=make_tooltip(grof, color_grof, "Grof"),
        ).add_to(layer_grof)

    if fijn > 0:
        color_fijn = to_hex_fijn(fijn)
        CircleMarker(
            location=[lat + 0.00003, lon + 0.00003],
            radius=6,
            color=color_fijn,
            fill=True,
            fill_color=color_fijn,
            fill_opacity=0.85,
            weight=0.8,
            tooltip=make_tooltip(fijn, color_fijn, "Fijn"),
        ).add_to(layer_fijn)

layer_grof.add_to(m)
layer_fijn.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

print(f"{missing_img} points had no matching image")

# ── 8. Legend ─────────────────────────────────────────────────────────────────

grof_min = int(points_wgs["Zwerfafval (grof)"][points_wgs["Zwerfafval (grof)"] > 0].min())
grof_max = int(points_wgs["Zwerfafval (grof)"].max())
grof_mid = round((grof_min + grof_max) / 2)

fijn_min = int(points_wgs["Zwerfafval (fijn)"][points_wgs["Zwerfafval (fijn)"] > 0].min())
fijn_max = int(points_wgs["Zwerfafval (fijn)"].max())
fijn_mid = round((fijn_min + fijn_max) / 2)

legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px">
  <div style="font-weight:600;margin-bottom:10px;color:#111">Detectiedichtheid</div>

  <div style="font-size:11px;font-weight:600;color:#444;margin-bottom:4px">Grof</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#ffffb2,#fecc5c,#fd8d3c,#f03b20,#bd0026)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:3px;color:#666;font-size:10px">
    <span>{grof_min}</span><span>{grof_mid}</span><span>{grof_max}</span>
  </div>

  <div style="font-size:11px;font-weight:600;color:#444;margin:12px 0 4px">Fijn</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#ffffd9,#c7e9b4,#41b6c4,#2c7fb8,#253494)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:3px;color:#666;font-size:10px">
    <span>{fijn_min}</span><span>{fijn_mid}</span><span>{fijn_max}</span>
  </div>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))
# ── 9. Save ───────────────────────────────────────────────────────────────────

out_path = os.path.join(
    "../datasets/experiments/zwerfafval",
    f"heatmap_dots_{model}_{split}.html"
)
m.save(out_path)
print(f"Saved → {out_path}  ({len(points_wgs)} points)")

0 points had no matching image
Saved → ../datasets/experiments/zwerfafval/heatmap_dots_yolo26m_1920_v1-2_extra_250-2_train.html  (160 points)


In [ ]:
# Andere set van images, Centrum 2026-07-13
#-------------------------------------------------------
predictions_folder1 = f"../datasets/experiments/zwerfafval/predict/inwinning_260713_26m_v1e2"
predictions_gdf1 = read_annotations_folder(folder_path=predictions_folder1, categories=categories)
predictions_gdf1["file_name"] = predictions_gdf1["file_name"].str.replace(".txt", ".jpg")

In [ ]:
_predictions_sorted = (
    predictions_gdf1[predictions_gdf1["confidence"] >= confidence]
    .set_index("file_name")
    .sort_index()
)

counts_df = (
    _predictions_sorted[["category"]]
    .replace(categories)
    .groupby(["file_name", "category"])
    .size()
    .unstack(fill_value=0)
)